In [ ]:
# Installiamo la versione di TRL che funzionava col vecchio codice
!pip install -q trl==0.8.1 transformers accelerate peft bitsandbytes datasets

In [ ]:
import torch
import pandas as pd
import os
import shutil
from IPython.display import FileLink
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from datasets import Dataset

# --- CONFIGURAZIONE ---
MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"
# Metti qui il percorso della cartella output del notebook 03
REWARD_MODEL_PATH = "/kaggle/input/datasets/silvioliparoti/deberta-weights-fin/deberta_judge_PPO_v2_fin"
DATASET_PATH = "/kaggle/input/dataset-ppo-v2/dataset_LPR_new_PPO_v2.csv"

# FORZATURA DISPOSITIVI
ZEPHYR_DEVICE = "cuda" # Zephyr DEVE stare su GPU
REWARD_DEVICE = "cpu"  # DeBERTa DEVE stare su CPU

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 2. Configurazione LoRA (Addestriamo solo il 2% dei parametri)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    peft_config=lora_config,
    device_map=ZEPHYR_DEVICE
)
''''
# --- IL FIX MAGICO ---
# Forziamo i layer di output a usare float32 per evitare i NaN
for name, module in model.named_modules():
    if "v_head" in name:
        module.to(torch.float32)
'''
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Caricamento Giudice su CPU per evitare i CUDA crashes")
reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_PATH,
    num_labels=1,
    torch_dtype=torch.float32
).to(REWARD_DEVICE)

reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_PATH)
print("Reward Model caricato (su CPU)!")

is_on_cpu = next(reward_model.parameters()).device.type == 'cpu'
if is_on_cpu:
    print("DeBERTa è sulla CPU.")
else:
    print("DeBERTa è ancora su GPU")

In [ ]:
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=['jokeText'])
df = df[df['jokeText'].str.len() > 5]
# Prendiamo le battute originali come prompt
dataset = Dataset.from_dict({"query": df['jokeText'].tolist()[:800]})

def tokenize(sample):
    sample["input_ids"] = tokenizer.encode(sample["query"], truncation=True, max_length=64)
    return sample

dataset = dataset.map(tokenize, batched=False)
dataset.set_format(type="torch")

def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

In [ ]:
config = PPOConfig(
    learning_rate=1e-6,
    batch_size=32,
    mini_batch_size=4,
    gradient_accumulation_steps=8,
    init_kl_coef=0.2,
    max_grad_norm=0.5,        # <--- TAGLIA GLI ERRORI PRIMA CHE ESPLODANO
    ppo_epochs=1              # <--- PIÙ CALMO NELL'AGGIORNAMENTO
)

ppo_trainer = PPOTrainer(
    config=config,
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator
)

In [ ]:
generation_kwargs = {
    "min_length": 5,
    "top_k": 50, # Intero
    "top_p": 0.9,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 32,
}

print(" Inizio PPO ...")

for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    try:
        query_tensors = batch["input_ids"]

        # 1. Generazione (GPU)
        response_tensors = ppo_trainer.generate(query_tensors, **generation_kwargs)
        batch["response"] = [tokenizer.decode(r.squeeze(), skip_special_tokens=True) for r in response_tensors]

        # 2. Reward (CPU)
        texts = [q + r for q, r in zip(batch["query"], batch["response"])]
        
        inputs = reward_tokenizer(
            texts, 
            padding=True, 
            truncation=True, 
            max_length=256, 
            return_tensors="pt"
        ).to(REWARD_DEVICE) 
        
        with torch.no_grad():
            outputs = reward_model(**inputs)
            rewards = [torch.tensor(val.item(), dtype=torch.float32, device="cpu") for val in outputs.logits]

        # 3. Step (GPU) - Ora funziona perché aggiorna LoRA, non il modello 4-bit
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        
        if epoch % 5 == 0:
            if len(rewards) > 0:
                avg = torch.mean(torch.stack(rewards))
                print(f"Iter {epoch}: Reward Medio = {avg:.4f}")
        
        torch.cuda.empty_cache()

    except Exception as e:
        print(f" Errore all'iterazione {epoch}: {e}")
        torch.cuda.empty_cache()
        continue

In [ ]:
# 1. Nome della cartella
output_dir = "zephyr-7b-ppo-borda-v2"

# 2. FIX FONDAMENTALE: Crea la cartella se non c'è 
os.makedirs(output_dir, exist_ok=True)

# 3. Salvataggio
print(" Salvataggio in corso...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(" Modello e Tokenizer salvati!")

# 4. Zippa tutto per scaricarlo 
shutil.make_archive(output_dir, 'zip', output_dir)

# 5. Genera il link per il download
FileLink(f"{output_dir}.zip")